This is the notebook in which I will do the pull from the Schiphol Airport API; for the midterm project, I analyzed a day's worth of departing flight information. For the final project, I'm going to analyze a month's worth of both arriving and departing flight information. I wanted to pull a year's worth, but at the speed the API can handle requests, it would've taken nearly 2 days. I used ChatGPT to help me formulate code to perform the pull because this API is paginated, rate-limited, and organized by day, so I needed a bunch of complicated for-loops to pull data from a wider time frame without triggering 429 errors.

In [2]:
# ============================================================
# Schiphol API Pull — Last 30 Complete Days
# Safe version: pulls into Pandas DataFrames, then saves one clean CSV
# ============================================================

import os
import time
import getpass
import random
import requests
import pandas as pd

from email.utils import parsedate_to_datetime
from datetime import datetime, timezone


# -----------------------------
# 1. Set up API credentials
# -----------------------------

app_id = os.getenv("SCHIPHOL_APP_ID")
app_key = os.getenv("SCHIPHOL_APP_KEY")

if app_id is None:
    app_id = getpass.getpass("Enter your Schiphol app_id: ")

if app_key is None:
    app_key = getpass.getpass("Enter your Schiphol app_key: ")


# -----------------------------
# 2. API setup
# -----------------------------

BASE_URL = "https://api.schiphol.nl/public-flights/flights"

headers = {
    "Accept": "application/json",
    "ResourceVersion": "v4",
    "app_id": app_id,
    "app_key": app_key
}


# -----------------------------
# 3. Choose date range: last 30 complete days
# -----------------------------

END_DATE = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
START_DATE = END_DATE - pd.Timedelta(days=29)

date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="D")

print(f"Pulling data from {START_DATE.date()} to {END_DATE.date()}")
print(f"Number of dates: {len(date_range)}")


# -----------------------------
# 4. Output and checkpoint settings
# -----------------------------

OUTPUT_CSV_FILE = "schiphol_arrivals_departures_last_month_clean.csv"
CHECKPOINT_FILE = "schiphol_arrivals_departures_last_month_chunks.pkl"
PROGRESS_FILE = "schiphol_arrivals_departures_last_month_progress.csv"

PAGE_SLEEP_SECONDS = 0.6
MAX_RETRIES = 6

# Set to True only if you intentionally want to restart the pull from scratch.
OVERWRITE_EXISTING_PULL = False


if OVERWRITE_EXISTING_PULL:
    for file in [OUTPUT_CSV_FILE, CHECKPOINT_FILE, PROGRESS_FILE]:
        if os.path.exists(file):
            os.remove(file)
            print(f"Deleted old file: {file}")


# -----------------------------
# 5. Helper function for Retry-After
# -----------------------------

def get_retry_after_seconds(response, default_wait):
    """
    Reads the Retry-After header if available.
    Retry-After can be a number of seconds or a date.
    """

    retry_after = response.headers.get("Retry-After")

    if retry_after is None:
        return default_wait

    try:
        return int(retry_after)
    except ValueError:
        pass

    try:
        retry_datetime = parsedate_to_datetime(retry_after)

        if retry_datetime.tzinfo is None:
            retry_datetime = retry_datetime.replace(tzinfo=timezone.utc)

        now = datetime.now(timezone.utc)
        wait_seconds = (retry_datetime - now).total_seconds()

        return max(1, int(wait_seconds))

    except Exception:
        return default_wait


# -----------------------------
# 6. Function to pull one date and direction
# -----------------------------

def fetch_schiphol_flights_one_day(schedule_date, flight_direction):
    """
    Pulls all Schiphol flight records for one date and one direction.

    flight_direction:
    A = arrivals
    D = departures
    """

    all_flights = []
    page = 0

    session = requests.Session()

    while True:
        params = {
            "scheduleDate": schedule_date,
            "flightDirection": flight_direction,
            "page": page
        }

        response = None

        for attempt in range(MAX_RETRIES):
            try:
                response = session.get(
                    BASE_URL,
                    headers=headers,
                    params=params,
                    timeout=(10, 90)
                )

            except requests.exceptions.ReadTimeout:
                wait_time = min(60, 5 * (2 ** attempt)) + random.uniform(0, 1)
                print(
                    f"Read timeout on {schedule_date}, direction {flight_direction}, "
                    f"page {page}. Waiting {wait_time:.1f} seconds..."
                )
                time.sleep(wait_time)
                continue

            except requests.exceptions.ConnectionError as e:
                wait_time = min(60, 5 * (2 ** attempt)) + random.uniform(0, 1)
                print(
                    f"Connection error on {schedule_date}, direction {flight_direction}, "
                    f"page {page}: {e}. Waiting {wait_time:.1f} seconds..."
                )
                time.sleep(wait_time)
                continue

            except requests.exceptions.RequestException as e:
                wait_time = min(60, 5 * (2 ** attempt)) + random.uniform(0, 1)
                print(
                    f"Request issue on {schedule_date}, direction {flight_direction}, "
                    f"page {page}: {e}. Waiting {wait_time:.1f} seconds..."
                )
                time.sleep(wait_time)
                continue

            # 429 = rate limited. Wait, then retry the same page.
            if response.status_code == 429:
                default_wait = min(60, 5 * (2 ** attempt))
                wait_time = get_retry_after_seconds(response, default_wait)
                wait_time = wait_time + random.uniform(0, 1)

                print(
                    f"429 rate limit on {schedule_date}, direction {flight_direction}, "
                    f"page {page}. Waiting {wait_time:.1f} seconds..."
                )

                time.sleep(wait_time)
                continue

            # Temporary server issue. Wait, then retry the same page.
            if response.status_code in [500, 502, 503, 504]:
                wait_time = min(60, 5 * (2 ** attempt)) + random.uniform(0, 1)

                print(
                    f"Server error {response.status_code} on {schedule_date}, "
                    f"direction {flight_direction}, page {page}. "
                    f"Waiting {wait_time:.1f} seconds..."
                )

                time.sleep(wait_time)
                continue

            break

        if response is None:
            session.close()
            raise RuntimeError(
                f"No response received for {schedule_date}, "
                f"direction {flight_direction}, page {page}"
            )

        # 204 means no more content / end of pages.
        if response.status_code == 204:
            if page == 0:
                print(f"No data available for {schedule_date}, direction {flight_direction}")
            else:
                print(f"Finished {schedule_date}, direction {flight_direction} after page {page - 1}")
            break

        if response.status_code != 200:
            session.close()
            raise RuntimeError(
                f"Request failed for {schedule_date}, direction {flight_direction}, "
                f"page {page}. Status code: {response.status_code}. "
                f"Response: {response.text[:500]}"
            )

        data = response.json()
        flights = data.get("flights", [])

        if len(flights) == 0:
            print(f"No more flights for {schedule_date}, direction {flight_direction}, page {page}")
            break

        all_flights.extend(flights)

        if page % 25 == 0:
            print(
                f"Still pulling {schedule_date}, direction {flight_direction}: "
                f"page {page}, rows so far {len(all_flights):,}"
            )

        page += 1
        time.sleep(PAGE_SLEEP_SECONDS)

    session.close()

    if len(all_flights) == 0:
        return pd.DataFrame()

    # json_normalize handles nested API fields better than pd.DataFrame.
    df = pd.json_normalize(all_flights)

    df["api_query_date"] = schedule_date
    df["flight_direction"] = flight_direction
    df["direction_label"] = df["flight_direction"].map({
        "A": "Arrival",
        "D": "Departure"
    })

    return df


# -----------------------------
# 7. Build full task list
# -----------------------------

all_tasks = []

for date in date_range:
    schedule_date = date.strftime("%Y-%m-%d")
    all_tasks.append((schedule_date, "A"))
    all_tasks.append((schedule_date, "D"))

print(f"Total date/direction tasks: {len(all_tasks)}")


# -----------------------------
# 8. Load existing checkpoint/progress if available
# -----------------------------

if os.path.exists(CHECKPOINT_FILE):
    all_flight_data = pd.read_pickle(CHECKPOINT_FILE)
    print(f"Loaded checkpoint with {len(all_flight_data)} completed data chunks.")
else:
    all_flight_data = []
    print("No checkpoint found. Starting with an empty data list.")


if os.path.exists(PROGRESS_FILE) and os.path.getsize(PROGRESS_FILE) > 0:
    progress_data = pd.read_csv(
        PROGRESS_FILE,
        dtype={"api_query_date": str, "flight_direction": str}
    )

    progress_data["api_query_date"] = pd.to_datetime(
        progress_data["api_query_date"],
        errors="coerce"
    ).dt.strftime("%Y-%m-%d")

    progress_data["flight_direction"] = (
        progress_data["flight_direction"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    completed_tasks = set(
        zip(progress_data["api_query_date"], progress_data["flight_direction"])
    )

    print(f"Loaded progress file with {len(completed_tasks)} completed tasks.")

else:
    completed_tasks = set()
    print("No progress file found.")


remaining_tasks = [
    task for task in all_tasks
    if task not in completed_tasks
]

print(f"Remaining tasks to pull: {len(remaining_tasks)}")


# -----------------------------
# 9. Pull remaining tasks directly into DataFrames
# -----------------------------

failed_tasks = []

total_tasks_this_run = len(remaining_tasks)
completed_this_run = 0
pull_start_time = time.time()

for schedule_date, direction in remaining_tasks:
    direction_name = "arrivals" if direction == "A" else "departures"

    print(f"\nPulling {schedule_date} - {direction_name}")

    try:
        daily_df = fetch_schiphol_flights_one_day(
            schedule_date=schedule_date,
            flight_direction=direction
        )

        if daily_df.empty:
            print(f"No data returned for {schedule_date} - {direction_name}")
            rows_pulled = 0

        else:
            all_flight_data.append(daily_df)
            rows_pulled = len(daily_df)
            print(f"Pulled {rows_pulled:,} rows for {schedule_date} - {direction_name}")

            # Save checkpoint after each successful data chunk.
            # This is NOT a CSV append, so it will not create malformed columns.
            pd.to_pickle(all_flight_data, CHECKPOINT_FILE)

        # Mark the task complete after it finishes successfully.
        progress_row = pd.DataFrame([{
            "api_query_date": schedule_date,
            "flight_direction": direction,
            "direction_name": direction_name,
            "rows_pulled": rows_pulled,
            "completed_at": pd.Timestamp.now()
        }])

        progress_row.to_csv(
            PROGRESS_FILE,
            mode="a",
            index=False,
            header=not os.path.exists(PROGRESS_FILE)
        )

    except Exception as e:
        print(f"Failed to pull {schedule_date} - {direction_name}: {e}")
        failed_tasks.append((schedule_date, direction, str(e)))

    completed_this_run += 1

    elapsed_minutes = (time.time() - pull_start_time) / 60

    if elapsed_minutes > 0:
        tasks_per_minute = completed_this_run / elapsed_minutes
        remaining_count = total_tasks_this_run - completed_this_run
        eta_minutes = remaining_count / tasks_per_minute if tasks_per_minute > 0 else None

        print(
            f"Progress: {completed_this_run}/{total_tasks_this_run} tasks complete | "
            f"Elapsed: {elapsed_minutes:.1f} min | "
            f"Rate: {tasks_per_minute:.2f} tasks/min | "
            f"ETA: {eta_minutes / 60:.2f} hours remaining"
        )

    time.sleep(0.5)


# -----------------------------
# 10. Combine all chunks safely in Pandas
# -----------------------------

if len(all_flight_data) == 0:
    raise ValueError("No flight data was pulled. Check API credentials, date range, or API responses.")

schiphol_flights = pd.concat(
    all_flight_data,
    ignore_index=True,
    sort=False
)

print("\nCombined dataframe shape:")
print(schiphol_flights.shape)

display(schiphol_flights.head())


# -----------------------------
# 11. Save one clean CSV at the end
# -----------------------------

schiphol_flights.to_csv(
    OUTPUT_CSV_FILE,
    index=False
)

print(f"\nSaved clean CSV: {OUTPUT_CSV_FILE}")


# -----------------------------
# 12. Verify the saved CSV can be read cleanly
# -----------------------------

test_read = pd.read_csv(OUTPUT_CSV_FILE, low_memory=False)

print("\nVerification read successful.")
print("Verified CSV shape:")
print(test_read.shape)


# -----------------------------
# 13. Quick checks
# -----------------------------

print("\nRows by direction:")
display(
    schiphol_flights["direction_label"]
    .value_counts(dropna=False)
    .reset_index(name="flight_count")
    .rename(columns={"direction_label": "direction"})
)

print("\nRows by API query date:")
display(
    schiphol_flights
    .groupby("api_query_date")
    .size()
    .reset_index(name="flight_count")
    .sort_values("api_query_date")
)

print("\nRows by date and direction:")
display(
    schiphol_flights
    .groupby(["api_query_date", "direction_label"])
    .size()
    .reset_index(name="flight_count")
    .sort_values(["api_query_date", "direction_label"])
)

if failed_tasks:
    print("\nFailed tasks:")
    for task in failed_tasks:
        print(task)
else:
    print("\nNo failed tasks.")

Enter your Schiphol app_id:  ········
Enter your Schiphol app_key:  ········


Pulling data from 2026-04-29 to 2026-05-28
Number of dates: 30
Total date/direction tasks: 60
No checkpoint found. Starting with an empty data list.
No progress file found.
Remaining tasks to pull: 60

Pulling 2026-04-29 - arrivals
Still pulling 2026-04-29, direction A: page 0, rows so far 20
Still pulling 2026-04-29, direction A: page 25, rows so far 520
Still pulling 2026-04-29, direction A: page 50, rows so far 1,020
Still pulling 2026-04-29, direction A: page 75, rows so far 1,520
Still pulling 2026-04-29, direction A: page 100, rows so far 2,020
Finished 2026-04-29, direction A after page 122
Pulled 2,456 rows for 2026-04-29 - arrivals
Progress: 1/60 tasks complete | Elapsed: 1.6 min | Rate: 0.63 tasks/min | ETA: 1.57 hours remaining

Pulling 2026-04-29 - departures
Still pulling 2026-04-29, direction D: page 0, rows so far 20
Still pulling 2026-04-29, direction D: page 25, rows so far 520
Still pulling 2026-04-29, direction D: page 50, rows so far 1,020
Still pulling 2026-04-29, 

,lastUpdatedAt,actualLandingTime,aircraftRegistration,estimatedLandingTime,expectedTimeOnBelt,flightDirection,flightName,flightNumber,gate,pier,...,api_query_date,flight_direction,direction_label,actualOffBlockTime,checkinAllocations.checkinAllocations,expectedTimeBoarding,expectedTimeGateClosing,expectedTimeGateOpen,publicEstimatedOffBlockTime,transferPositions.transferPositions
0,2026-04-29T02:00:59.085+02:00,2026-04-29T00:20:21.000+02:00,PHYHY,2026-04-29T00:20:21.000+02:00,2026-04-29T01:26:39.433+02:00,A,HV5960,5960,Z02,Z,...,2026-04-29,A,Arrival,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-29T02:54:10.101+02:00,2026-04-29T00:16:49.000+02:00,PHHBK,2026-04-29T00:16:49.000+02:00,2026-04-29T00:59:10.309+02:00,A,HV6706,6706,D84,D,...,2026-04-29,A,Arrival,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-29T01:11:06.081+02:00,2026-04-28T23:38:34.000+02:00,PHHZI,2026-04-28T23:38:34.000+02:00,2026-04-29T00:07:29.000+02:00,A,HV6862,6862,D81,D,...,2026-04-29,A,Arrival,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-29T01:57:50.092+02:00,2026-04-29T00:33:43.000+02:00,PHHSB,2026-04-29T00:33:43.000+02:00,2026-04-29T00:54:42.000+02:00,A,HV5134,5134,C5,C,...,2026-04-29,A,Arrival,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-29T01:48:06.083+02:00,2026-04-29T00:03:52.000+02:00,PHYHU,2026-04-29T00:03:52.000+02:00,2026-04-29T00:38:31.000+02:00,A,HV5666,5666,Z07,Z,...,2026-04-29,A,Arrival,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved clean CSV: schiphol_arrivals_departures_last_month_clean.csv

Verification read successful.
Verified CSV shape:
(146563, 41)

Rows by direction:


,direction,flight_count
0,Arrival,73599
1,Departure,72964



Rows by API query date:


,api_query_date,flight_count
0,2026-04-29,4879
1,2026-04-30,4902
2,2026-05-01,4952
3,2026-05-02,4734
4,2026-05-03,4807
5,2026-05-04,4944
6,2026-05-05,4853
7,2026-05-06,4854
8,2026-05-07,4898
9,2026-05-08,4928



Rows by date and direction:


,api_query_date,direction_label,flight_count
0,2026-04-29,Arrival,2456
1,2026-04-29,Departure,2423
2,2026-04-30,Arrival,2465
3,2026-04-30,Departure,2437
4,2026-05-01,Arrival,2483
5,2026-05-01,Departure,2469
6,2026-05-02,Arrival,2376
7,2026-05-02,Departure,2358
8,2026-05-03,Arrival,2411
9,2026-05-03,Departure,2396



No failed tasks.
